# Tier 1 — Modèles prêts à l'emploi : Détection visage + Âge + Sexe

**Projet** : Analyse faciale pour recommandation parapharmaceutique
**Phase** : Tier 1 (aucun entraînement — modèles pré-entraînés)
**Environnement** : Kaggle Notebook

Ce notebook met en place la première brique du pipeline :
1. Détection + alignement du visage
2. Estimation de l'**âge**
3. Classification du **sexe**

avec deux bibliothèques matures (**DeepFace** et **InsightFace**), puis une **fonction unifiée**
renvoyant une sortie structurée (JSON) directement réutilisable plus tard dans le backend FastAPI.

> ⚙️ **Avant de lancer** : dans le panneau de droite Kaggle →
> - **Settings → Accelerator : GPU** (T4 recommandé)
> - **Settings → Internet : ON** (obligatoire : les poids des modèles se téléchargent au 1er appel)


## 1. Installation des dépendances

Sur Kaggle, l'image de base contient déjà TensorFlow/PyTorch. On ajoute :
- `deepface` (+ `tf-keras` requis par les versions récentes)
- `insightface` + `onnxruntime-gpu` (inférence ONNX)


In [ ]:
!pip install -q deepface tf-keras insightface onnxruntime-gpu
!pip install -q --force-reinstall "numpy==1.26.4" "scipy==1.13.1"
import IPython
print("Redémarrage du kernel… ensuite, reprends à partir de la Section 2 (Imports).")
IPython.Application.instance().kernel.do_shutdown(restart=True)

: 

## 2. Imports & configuration

In [ ]:
import os
import glob
import json
import warnings
warnings.filterwarnings("ignore")

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Vérification GPU
import onnxruntime as ort
print("Providers ONNX disponibles :", ort.get_available_providers())

USE_GPU = "CUDAExecutionProvider" in ort.get_available_providers()
print("GPU utilisé :", USE_GPU)

from insightface.app import FaceAnalysis

providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] if USE_GPU else ["CPUExecutionProvider"]

face_app = FaceAnalysis(name="buffalo_l", providers=providers)
face_app.prepare(ctx_id=0 if USE_GPU else -1, det_size=(640, 640))
print("InsightFace prêt.")

## 3. Préparation des images

Sur Kaggle, les jeux de données montés se trouvent sous `/kaggle/input/...`
et les sorties dans `/kaggle/working/`.

👉 Ajoute tes images de test via **+ Add Input** (un dataset d'images de visages),
le notebook les détecte automatiquement.


In [ ]:
INPUT_DIR  = "/kaggle/input"
OUTPUT_DIR = "/kaggle/working"

# Recherche récursive des images dans /kaggle/input
exts = ("*.jpg", "*.jpeg", "*.png", "*.webp")
images = []
for ext in exts:
    images += glob.glob(os.path.join(INPUT_DIR, "**", ext), recursive=True)

images = sorted(images)
print(f"{len(images)} image(s) trouvée(s).")
for p in images[:5]:
    print(" -", p)

if not images:
    print("\n⚠️  Aucune image. Ajoute un dataset via '+ Add Input' puis relance cette cellule.")

def load_image_rgb(path):
    """Charge une image en RGB (OpenCV lit en BGR par défaut)."""
    img = cv2.imread(path)
    if img is None:
        raise ValueError(f"Image illisible : {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Image d'exemple pour les tests ci-dessous
sample_path = images[0] if images else None
print("Image d'exemple :", sample_path)

## 4. Détection + alignement du visage (InsightFace)

InsightFace fournit un détecteur **SCRFD** rapide et robuste, l'alignement,
et — dans le même modèle `buffalo_l` — les têtes **âge** et **genre**.
On initialise l'analyseur une seule fois (les poids se téléchargent au 1er appel).


In [ ]:

def analyze_insightface(path):
    """Renvoie une liste de dicts {age, sexe, bbox, score} par visage."""
    _, faces = detect_faces_insightface(path)
    out = []
    for f in faces:
        # f.gender : 1 = homme, 0 = femme ; f.age : entier
        sexe = "Homme" if int(f.gender) == 1 else "Femme"
        out.append({
            "age": int(f.age),
            "sexe": sexe,
            "bbox": f.bbox.astype(int).tolist(),
            "score": float(f.det_score),
        })
    return out

if sample_path:
    print("InsightFace :", analyze_insightface(sample_path))

## 7. Fonction unifiée (sortie structurée)

On encapsule le tout dans **une fonction réutilisable** qui renvoie un dictionnaire
au format aligné avec le futur profil client (champ `demographics`).
C'est cette fonction que tu porteras quasi telle quelle dans l'endpoint FastAPI `/v1/analyze`.

`engine` permet de choisir le moteur ; InsightFace par défaut.


In [ ]:
def analyze_demographics(path, engine="insightface"):
    """Analyse Tier 1 d'une image -> dict structuré (1er visage = principal)."""
    if engine == "deepface":
        faces = analyze_deepface(path)
        faces = [{"age": f["age"], "sexe": f["sexe"]} for f in faces]
    else:
        faces = analyze_insightface(path)

    if not faces:
        return {"ok": False, "raison": "aucun_visage_detecte", "image": os.path.basename(path)}

    principal = faces[0]  # visage principal = premier détecté
    return {
        "ok": True,
        "image": os.path.basename(path),
        "moteur": engine,
        "nb_visages": len(faces),
        "demographics": {
            "age_estime": principal["age"],
            "sexe": principal["sexe"],
        },
    }

if sample_path:
    profil = analyze_demographics(sample_path, engine="insightface")
    print(json.dumps(profil, indent=2, ensure_ascii=False))

## 8. Visualisation

On affiche l'image avec la boîte du visage et le résultat (âge / sexe) en surimpression.


In [ ]:
def show_result(path, engine="insightface"):
    img_rgb = load_image_rgb(path)
    faces = analyze_insightface(path)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(img_rgb)
    ax.axis("off")

    for f in faces:
        x1, y1, x2, y2 = f["bbox"]
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                    fill=False, edgecolor="#1ABC9C", linewidth=2))
        ax.text(x1, max(y1 - 8, 0), f"{f['sexe']}, {f['age']} ans",
                color="white", fontsize=11,
                bbox=dict(facecolor="#1ABC9C", edgecolor="none", pad=2))
    plt.tight_layout()
    plt.show()

if sample_path:
    show_result(sample_path)

## Live demo


In [ ]:
import numpy as np

# buffalo_l : face.pose = [pitch, yaw, roll] (degrés)
# pitch>0 = haut | yaw>0 = tourné à GAUCHE | roll>0 = incliné à gauche
def get_pose(face):
    p = getattr(face, "pose", None)
    if p is None:
        return None
    p = np.asarray(p, dtype=float).ravel()
    return {"pitch": float(p[0]), "yaw": float(p[1]), "roll": float(p[2])}

# Seuils par vue (à affiner sur tes propres images)
VIEWS = {
    "face":          {"yaw": (-12, 12),  "pitch_tol": 14, "roll_tol": 14, "requires_face": True},
    "profil_gauche": {"yaw": (30, 58),   "pitch_tol": 20, "roll_tol": 18, "requires_face": True},
    "profil_droit":  {"yaw": (-58, -30), "pitch_tol": 20, "roll_tol": 18, "requires_face": True},
    "dessus":        {"requires_face": False},
}
MIN_FACE_RATIO, MAX_FACE_RATIO = 0.05, 0.60   # proxy de distance (surface bbox / image)

In [ ]:
def evaluate_pose(target, faces, img_shape):
    """Retourne (conforme: bool, hint: str) pour la vue 'target'."""
    cfg = VIEWS[target]
    H, W = img_shape[:2]

    # Vue du dessus : on veut justement NE PAS voir de visage frontal
    if not cfg["requires_face"]:
        if len(faces) == 0:
            return True, "Position correcte (aucun visage frontal)"
        return False, "Visage encore visible : penchez la tête en avant"

    if len(faces) == 0:
        return False, "Aucun visage détecté"
    if len(faces) > 1:
        return False, "Plusieurs visages détectés"

    face = faces[0]
    x1, y1, x2, y2 = face.bbox
    ratio = ((x2 - x1) * (y2 - y1)) / (W * H)
    if ratio < MIN_FACE_RATIO:
        return False, "Trop loin (rapprochez-vous)"
    if ratio > MAX_FACE_RATIO:
        return False, "Trop près (éloignez-vous)"

    pose = get_pose(face)
    if pose is None:
        return False, "Pose indisponible"
    if abs(pose["roll"]) > cfg["roll_tol"]:
        return False, "Gardez la tête droite"
    if abs(pose["pitch"]) > cfg["pitch_tol"]:
        return False, "Ne penchez pas la tête (haut/bas)"

    ymin, ymax = cfg["yaw"]
    if pose["yaw"] < ymin:
        return False, f"Tournez plus (yaw={pose['yaw']:.0f}°)"
    if pose["yaw"] > ymax:
        return False, f"Tournez moins (yaw={pose['yaw']:.0f}°)"

    return True, "Position correcte"

In [ ]:
def classify_view(faces):
    """Devine à quelle vue correspond une image."""
    if len(faces) == 0:
        return "dessus", "aucun visage → vue du dessus probable"
    pose = get_pose(faces[0])
    if pose is None:
        return "indetermine", "pose indisponible"
    yaw = pose["yaw"]
    if -12 <= yaw <= 12:
        return "face", f"yaw={yaw:.0f}°"
    if yaw > 12:
        return "profil_gauche", f"yaw={yaw:.0f}°"
    return "profil_droit", f"yaw={yaw:.0f}°"

In [ ]:
import cv2
import matplotlib.pyplot as plt

def show_pose_check(path, target):
    img_rgb = load_image_rgb(path)
    faces = face_app.get(cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))
    conforme, hint = evaluate_pose(target, faces, img_rgb.shape)
    color = (46, 204, 113) if conforme else (231, 76, 60)  # vert / rouge

    vis = img_rgb.copy()
    for f in faces:
        x1, y1, x2, y2 = f.bbox.astype(int)
        cv2.rectangle(vis, (x1, y1), (x2, y2), color, 3)
        pose = get_pose(f)
        if pose:
            cv2.putText(vis, f"yaw={pose['yaw']:.0f} pitch={pose['pitch']:.0f}",
                        (x1, max(y1 - 10, 20)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

    plt.figure(figsize=(6, 6)); plt.imshow(vis); plt.axis("off")
    plt.title(f"Cible: {target}  →  {'✓ ' if conforme else '✗ '}{hint}",
              color="green" if conforme else "red")
    plt.show()

if sample_path:
    show_pose_check(sample_path, target="face")

In [ ]:
import pandas as pd

rows = []
for p in images:
    faces = face_app.get(cv2.cvtColor(load_image_rgb(p), cv2.COLOR_RGB2BGR))
    vue, info = classify_view(faces)
    pose = get_pose(faces[0]) if faces else None
    rows.append({
        "image": os.path.basename(p),
        "vue_detectee": vue,
        "yaw": round(pose["yaw"], 1) if pose else None,
        "pitch": round(pose["pitch"], 1) if pose else None,
        "roll": round(pose["roll"], 1) if pose else None,
    })

pd.DataFrame(rows)